# **Installing Libraries and Packages**



***Please make note that for this project chatgpt was used to help with generating synthetic data, editing some of the prompts and conversational section heavily. Overall ChatGPT was used etensively to write this code.***

In [1]:
# =========================
# Cell 1: installs & imports
# =========================

!pip install -q "transformers==4.28.1" accelerate huggingface_hub sentencepiece datasets pandas

import torch
from transformers import LlamaTokenizer, LlamaForCausalLM
from dataclasses import dataclass
from typing import List, Dict, Any
import re
import json
import pandas as pd

# =========================
# Load PMC-LLaMA 13B
# =========================

# NOTE: This requires a GPU with a decent amount of VRAM (e.g., Colab T4/A100).
# If you hit OOM, switch to a smaller LLaMA variant or another model.

tokenizer = LlamaTokenizer.from_pretrained("axiong/PMC_LLaMA_13B")

model = LlamaForCausalLM.from_pretrained(
    "axiong/PMC_LLaMA_13B",
    torch_dtype=torch.float16,
    device_map="auto"
)

# def llama_generate(prompt: str, max_new_tokens: int = 256) -> str:
#     """
#     Wrapper around PMC-LLaMA generation.
#     Deterministic (temperature=0.0) for reproducible evaluations.
#     """
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         temperature=0.0,
#         do_sample=False
#     )
#     return tokenizer.decode(outputs[0], skip_special_tokens=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 25.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 152.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

pytorch_model-00004-of-00006.bin:   0%|          | 0.00/9.87G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [2]:


# If you already loaded tokenizer/model, don't reload; just redefine the function.
# Assuming:
# tokenizer = transformers.LlamaTokenizer.from_pretrained("axiong/PMC_LLaMA_13B")
# model = transformers.LlamaForCausalLM.from_pretrained("axiong/PMC_LLaMA_13B").to(device)

def llama_generate(prompt: str, max_new_tokens: int = 256) -> str:
    """
    Generate ONLY the completion (without echoing the prompt).
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    input_len = input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # greedy
        )

    # Drop the prompt part, keep only new tokens
    gen_ids = outputs[0, input_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


In [3]:
# Three detailed rural-style patient descriptions + ground-truth labels

PATIENT_PROFILES = {
    "P001": {  # Acute myocardial infarction (heart attack)
        "description": (
            "I am a 62-year-old man living far from the city on a farm. Since this morning, "
            "I have had a heavy pressure in the middle of my chest that will not go away. "
            "It gets worse when I try to walk or do any work. The pain sometimes spreads to my left arm "
            "and up into my jaw. I feel sweaty and a little short of breath. I feel sick to my stomach "
            "but I have not thrown up. I do NOT have a fever. I have smoked for many years. "
            "My brother had heart problems before. I have never felt anything like this before. "
            "It is hard for me to reach a hospital quickly from where I live in this rural area."
        ),
        "label_diseases": ["acute myocardial infarction"],
    },
    "P002": {  # Panic disorder with recurrent panic attacks
        "description": (
            "I am a 24-year-old woman from a small town. For the past few weeks, I have had repeated episodes "
            "where I suddenly feel intense fear and anxiety out of nowhere. During these episodes my heart "
            "starts pounding very fast and hard, I feel short of breath, and I sometimes have a tight or "
            "uncomfortable feeling in my chest. I feel like something terrible is about to happen or that I "
            "might die, even though it lasts only several minutes and then slowly gets better. "
            "Between these episodes, I worry a lot about having another one and I have started avoiding places "
            "where they happened before, like crowded stores. A doctor once told me my heart tests were normal. "
            "I do NOT have constant chest pain. I do NOT faint. I am not pregnant. I do not smoke or drink alcohol. "
            "I recently started a very stressful new job away from my family, and it is hard to access regular "
            "mental health care in my rural area."
        ),
        "label_diseases": ["panic disorder"],
    },
    "P003": {  # Acute appendicitis
        "description": (
            "I am a 17-year-old boy from a rural area. I started having stomach pain yesterday around my belly button, "
            "and today the pain has moved to the lower right side of my belly. It hurts more when I walk, cough, "
            "or when someone presses on that area. I feel nauseous and do not want to eat anything. "
            "I had a fever last night. I do NOT have burning when I pee. I do NOT have diarrhea. "
            "I have never had stomach surgery before. It would take a long time for me to get to the nearest emergency hospital."
        ),
        "label_diseases": ["acute appendicitis"],
    },
}


# **Zero-Shot Response**

In [6]:
def doctor_single_pass(note: str) -> str:
    """
    Single-pass 'doctor' that returns plain text with 3 sections.
    No JSON, no placeholders.
    """
    prompt = f"""
You are a cautious primary care doctor in a rural area with limited access to tests and specialists.

Here is a patient description written in the patient's own words:

\"\"\"{note}\"\"\"

Your tasks:
1. List the most likely diagnoses that explain these symptoms and history.
2. Suggest safe first-line medications and doses for each diagnosis in a rural primary-care setting.
3. Briefly discuss medication safety and when urgent in-person care or transfer to a hospital is needed.


"""
    return llama_generate(prompt, max_new_tokens=300)


In [7]:
results = []
final_outputs = {}

for pid, profile in PATIENT_PROFILES.items():
    print("\n====================")
    print(f"### Patient {pid}")
    note = profile["description"]

    out = doctor_single_pass(note)
    print(out)  # now you should see DIAGNOSES / MEDICATIONS / SAFETY only

    # final_outputs[pid] = out
    # results.append(evaluate_output(pid, out))

# df_results = pd.DataFrame(results)
# df_results



### Patient P001
###Suggested answers:

The patient's symptoms and history are suggestive of Unstable Angina (UA) or Non-ST Elevation Myocardial Infarction (NSTEMI). The first-line medications for UA/NSTEMI include antiplatelet agents (e.g., Aspirin), beta-blockers, nitrates, and statins. The choice of medications and their doses may vary based on the patient's comorbidities, renal function, and other factors.

In a rural primary-care setting, the first-line medications for UA/NSTEMI are:

- Aspirin: 325 mg loading dose followed by 81 mg daily
- Clopidogrel: 300 mg loading dose followed by 75 mg daily if the patient is unable to take aspirin
- Beta-blocker: Metoprolol 5-15 mg IV bolus followed by 5 mg IV infusion, then 25-50 mg orally every 6-8 hours
- Nitrate: Nitroglycerin 5-10 mg IV infusion, then 2-5 mg every 5 minutes until the patient's symptoms improve or the maximum do

### Patient P002
###Depending on the answers to the questions above, the patient may have:

Panic disorder
A

# **In-Context Learning**

In [17]:
def doctor_ict(note: str) -> str:
    prompt = f"""
You are a cautious primary care doctor in a rural area with limited access to tests and specialists.

Your output must ALWAYS include:
- At least **1 likely diagnosis**
- At least **1 safe first-line medication**
- A brief safety note (1–3 sentences)
- **No placeholders** like "...", "TBD", "<1-2 sentences>".
- Follow the format of the examples EXACTLY.

Here are examples to learn from:

EXAMPLE 1 — Asthma
Patient:
"I am a 19-year-old boy from a rural town. I have asthma since childhood. Yesterday after cleaning dusty rooms I feel chest tightness, coughing and wheezing. No chest pain. No fever. Hard to refill inhaler due to distance."

DIAGNOSES:
- Mild asthma exacerbation

MEDICATIONS:
- Albuterol inhaler, 2 puffs every 4–6 hours as needed – relieve wheezing
- Prednisone 40 mg, once daily for 5 days – reduce airway inflammation

SAFETY:
- If breathing worsens or lips turn blue, go to urgent care immediately.

---

Now follow the same style for this new patient:

Patient:
\"\"\"{note}\"\"\"


"""
    return llama_generate(prompt, max_new_tokens=350)


In [18]:
results = []
final_outputs = {}

for pid, profile in PATIENT_PROFILES.items():
    print("\n====================")
    print(f"### Patient {pid}")
    note = profile["description"]

    out = doctor_ict(note)
    print(out+"\n")  # now you should see DIAGNOSES / MEDICATIONS / SAFETY only



### Patient P001
###OUTPUT:

Your likely diagnosis:
- Unstable angina or non-ST elevation myocardial infarction (NSTEMI)

Your medications:
- Aspirin 325 mg, chew now – prevent further blood clotting
- Nitroglycerin 0.4 mg sublingual, as needed – relieve chest pressure
- Metoprolol 5 mg IV, as needed – reduce heart rate and blood pressure
- Oxygen via nasal cannula – increase oxygenation
- Morphine 2 mg IV, as needed – relieve pain
- ECG and cardiac enzymes – confirm the diagnosis
- Admission to the hospital for further monitoring and treatment

SAFETY:
- If symptoms worsen, go to the nearest hospital emergency room immediately.

###ANSWER: OPTION A IS CORRECT.


### Patient P002
###OUTPUT:

Diagnoses:
- Panic disorder

Medications:
- Clonazepam 0.5 mg, once daily at bedtime – reduce anxiety
- Sertraline 25 mg, once daily in the morning – reduce fear and anxiety

Safety:
- If symptoms worsen, seek urgent care for evaluation of heart disease or other conditions.
- Consider seeing a men

# **Chain of Thoughts**

In [19]:
def doctor_cot(note: str) -> str:
    prompt = f"""
You are a cautious primary care doctor in a rural area with limited access to tests and specialists.

Step 1: From the note, list key medical findings that look like diagnoses or chronic conditions.
Step 2: From those findings, choose which are clearly diagnosed diseases.
Step 3: Under 'Final diseases:', list those diseases one per line.
Step 4: Make sure that diagnoses align with patient age, location and symptoms.
Step 5: Suggest medications based on patient's rural area limitation.
Step 6: Based on diagnoses, suggest if patient needs to be moved to a hospital or specialist while giving temporary medication for relief.

Patient Description:
\"\"\"{note}\"\"\"

Begin reasoning.

Step 1 – Findings:
"""
    return llama_generate(prompt, max_new_tokens=200)



In [20]:
results = []
final_outputs = {}

for pid, profile in PATIENT_PROFILES.items():
    print("\n====================")
    print(f"### Patient {pid}")
    note = profile["description"]

    out = doctor_cot(note)
    print(out+"\n")  # now you should see DIAGNOSES / MEDICATIONS / SAFETY only



### Patient P001

Step 2 – Diagnoses:

Final diagnoses:

- Angina
- Myocardial infarction
- Unstable angina
- Acute coronary syndrome
- GERD
- Peptic ulcer disease
- Acute gastritis
- Musculoskeletal pain

Step 3 – Aligning diagnoses with symptoms:

- Angina and myocardial infarction align with chest pain
- Unstable angina aligns with shortness of breath and sweating
- GERD aligns with stomach pain and nausea
- Peptic ulcer disease aligns with stomach pain
- Acute gastritis aligns with stomach pain
- Musculoskeletal pain aligns with pain worsening with movement

Step 4 – Location and age:

- The patient's age


### Patient P002
- Sudden intense fear and anxiety
- Palpitations
- Shortness of breath
- Chest discomfort
- Worry about having another episode
- Avoidance

Step 2 – Choose which are clearly diagnosed diseases:
- Panic disorder
- Generalized anxiety disorder

Step 3 – Final diagnoses:
- Panic disorder
- Generalized anxiety disorder

Step 4 – Align diagnoses with patient age, lo

# **Tree of Thoughts**

In [35]:
def doctor_tot(note: str) -> str:
    """
    Tree-of-Thought style, but we DO NOT show the thought paths.
    We just ask the model to think step by step internally,
    then output a clean final block we can evaluate.
    """
    prompt = f"""
You are a cautious primary care doctor with years of experience in a rural area
with limited access to tests and specialists.

A patient describes their symptoms:

\"\"\"{note}\"\"\"

Your internal reasoning (do NOT write this out) should cover:
- Possible diseases based on the patient's symptoms and description.
- Whether the symptoms fit the patient's age and rural context.
- Which single diagnosis is MOST LIKELY overall.
- Whether the patient needs urgent hospitalization or referral to a specialist.
- What medications are reasonable in a rural primary-care setting
  (short-term if transferring, or longer-term if managed locally).

After you have thought this through SILENTLY, write ONLY the final answer.
Follow this exact format. Do NOT include your reasoning. Do NOT use placeholders like "...".
Do NOT say things like "Option A is correct".

DIAGNOSIS:
- <single most likely disease>

MEDICATIONS:
- <name>, <dose>, <frequency> – <purpose>
- <name>, <dose>, <frequency> – <purpose>

SAFETY:
- <1–3 sentences about urgency, transfer, or follow-up>

Now provide your final diagnoses:
"""
    return llama_generate(prompt, max_new_tokens=300)


In [36]:
results = []
final_outputs = {}

for pid, profile in PATIENT_PROFILES.items():
    print("\n====================")
    print(f"### Patient {pid}")
    note = profile["description"]

    out = doctor_tot(note)
    print(out+"\n")  # now you should see DIAGNOSES / MEDICATIONS / SAFETY only



### Patient P001

Option A: Acute myocardial infarction
- The patient's symptoms of chest pain radiating to the left arm and jaw, along with shortness of breath and nausea, are classic for acute myocardial infarction.
- The patient's age, history of smoking, and family history of heart problems further support this diagnosis.
- Given the severity of the symptoms and the patient's risk factors, urgent hospitalization is recommended.
- Treatment should include aspirin, nitroglycerin, and morphine, as well as possible transfer to a facility with cardiac interventions.

Option B: Angina pectoris
- Angina pectoris is a type of chest pain that occurs when the heart muscle does not receive enough oxygen-rich blood.
- While the patient's symptoms could be consistent with angina, the severity of the symptoms and the patient's risk factors make acute myocardial infarction a more likely diagnosis.
- Treatment for angina pectoris typically involves rest, nitroglycerin, and possibly aspirin, but h

# **Conversational Doctor**

In [41]:
# ============================================
# Cell 2: Patient profiles & conversational doctor (NO JSON)
# ============================================

from typing import List

# 3 rural-style patients (ground truth for evaluation)
PATIENT_PROFILES = {
    "P001": {  # Acute myocardial infarction (heart attack)
        "description": (
            "I am a 62-year-old man living far from the city on a farm. Since this morning, "
            "I have had a heavy pressure in the middle of my chest that will not go away. "
            "It gets worse when I try to walk or do any work. The pain sometimes spreads to my left arm "
            "and up into my jaw. I feel sweaty and a little short of breath. I feel sick to my stomach "
            "but I have not thrown up. I do NOT have a fever. I have smoked for many years. "
            "My brother had heart problems before. I have never felt anything like this before. "
            "It is hard for me to reach a hospital quickly from where I live in this rural area."
        ),
        "label_diseases": ["acute myocardial infarction"],
    },
    "P002": {  # Panic disorder with recurrent panic attacks
        "description": (
            "I am a 24-year-old woman from a small town. For the past few weeks, I have had repeated episodes "
            "where I suddenly feel intense fear and anxiety out of nowhere. During these episodes my heart "
            "starts pounding very fast and hard, I feel short of breath, and I sometimes have a tight or "
            "uncomfortable feeling in my chest. I feel like something terrible is about to happen or that I "
            "might die, even though it lasts only several minutes and then slowly gets better. "
            "Between these episodes, I worry a lot about having another one and I have started avoiding places "
            "where they happened before, like crowded stores. A doctor once told me my heart tests were normal. "
            "I do NOT have constant chest pain. I do NOT faint. I am not pregnant. I do not smoke or drink alcohol. "
            "I recently started a very stressful new job away from my family, and it is hard to access regular "
            "mental health care in my rural area."
        ),
        "label_diseases": ["panic disorder"],
    },
    "P003": {  # Acute appendicitis
        "description": (
            "I am a 17-year-old boy from a rural area. I started having stomach pain yesterday around my belly button, "
            "and today the pain has moved to the lower right side of my belly. It hurts more when I walk, cough, "
            "or when someone presses on that area. I feel nauseous and do not want to eat anything. "
            "I had a fever last night. I do NOT have burning when I pee. I do NOT have diarrhea. "
            "I have never had stomach surgery before. It would take a long time for me to get to the nearest emergency hospital."
        ),
        "label_diseases": ["acute appendicitis"],
    },
}

# Simple rule-based simulated answers
PATIENT_RESPONSES = {
    "P001": {  # MI
        "yes_no": {
            "chest pain": "Yes, it feels like a heavy pressure in the middle of my chest.",
            "arm": "Yes, it sometimes goes down my left arm.",
            "jaw": "Yes, it sometimes goes up into my jaw.",
            "shortness of breath": "Yes, I feel a little short of breath.",
            "sweat": "Yes, I feel sweaty and clammy.",
            "fever": "No, I do not have a fever.",
            "vomit": "No, I feel sick to my stomach but I have not thrown up.",
            "smoke": "Yes, I have smoked for many years.",
        },
        "general": {
            "how long": "It started this morning and has not gone away.",
            "family": "My brother had heart problems before.",
            "before": "I have never had pain like this before.",
        },
    },
    "P002": {  # Panic disorder
        "yes_no": {
            "chest pain": "I get a tight or uncomfortable feeling in my chest during the attacks.",
            "shortness of breath": "Yes, I feel short of breath during the episodes.",
            "palpitation": "Yes, my heart feels like it is pounding very fast.",
            "faint": "No, I do not actually faint.",
        },
        "general": {
            "how long": "The episodes started a few weeks ago.",
            "duration": "Each one lasts several minutes and then slowly gets better.",
            "stress": "I recently started a very stressful new job away from my family.",
            "avoid": "I have started avoiding places where the attacks happened before.",
        },
    },
    "P003": {  # Appendicitis
        "yes_no": {
            "fever": "Yes, I had a fever last night.",
            "urine": "No, my urine seems normal.",
            "back": "No, I do not have pain in my back.",
        },
        "general": {
            "where": "It started around my belly button and now it's mostly on the lower right side.",
            "worse": "It hurts more when I walk or someone presses there.",
            "eat": "I do not feel like eating anything.",
            "when": "The pain started yesterday and has gotten worse today.",
        },
    },
}




In [44]:
DOCTOR_SYSTEM_PROMPT = """
You are a cautious primary care doctor in a rural area.

Rules:
- Ask ONE medical question at a time.
- Do NOT give diagnoses yet.
- Do NOT give medications yet.
- Use simple language a patient understands.
"""


def simulate_patient_reply(patient_id: str, doctor_question: str) -> str:
    q = doctor_question.lower()
    qa = PATIENT_RESPONSES.get(patient_id, {})
    yes_no = qa.get("yes_no", {})
    general = qa.get("general", {})

    for kw, resp in yes_no.items():
        if kw in q:
            return resp
    for kw, resp in general.items():
        if kw in q:
            return resp
    return "I’m not sure how to answer that, but my main problem is what I already described."


def doctor_conversation(patient_id: str, max_turns: int = 5):
    if patient_id not in PATIENT_PROFILES:
        raise ValueError(f"Unknown patient_id: {patient_id}")

    patient_text = PATIENT_PROFILES[patient_id]["description"]
    conversation = f"PATIENT: {patient_text}\n"
    transcript: List[str] = []

    print("=== Conversation Start ===")
    print(patient_text)

    for _ in range(max_turns):
        prompt = DOCTOR_SYSTEM_PROMPT + "\n\nConversation so far:\n" + conversation + "DOCTOR: "
        doctor_output = llama_generate(prompt, max_new_tokens=64).strip()
        transcript.append("DOCTOR: " + doctor_output)
        print("Doctor:", doctor_output)

        # Stop if doctor tries to diagnose
        if "diagnosis" in doctor_output.lower():
            break

        # Patient replies
        reply = simulate_patient_reply(patient_id, doctor_output)
        transcript.append("PATIENT: " + reply)
        conversation += f"DOCTOR: {doctor_output}\nPATIENT: {reply}\n"
        print("Patient:", reply)

    # Final direct assessment (NO JSON)
    final_prompt = f"""
You are now allowed to give diagnoses and treatment.
Write clear medical advice.

Conversation:
{conversation}
"""
    final_output = llama_generate(final_prompt, max_new_tokens=200)
    transcript.append("DOCTOR FINAL: " + final_output)

    print("\n=== FINAL DOCTOR OUTPUT ===")
    print(final_output)

    return transcript, final_output

In [45]:
for pid in ["P001", "P002", "P003"]:
    print("\n==================")
    print(f"Running doctor on {pid}")
    transcript, final_output = doctor_conversation(pid, max_turns=5)



Running doctor on P001
=== Conversation Start ===
I am a 62-year-old man living far from the city on a farm. Since this morning, I have had a heavy pressure in the middle of my chest that will not go away. It gets worse when I try to walk or do any work. The pain sometimes spreads to my left arm and up into my jaw. I feel sweaty and a little short of breath. I feel sick to my stomach but I have not thrown up. I do NOT have a fever. I have smoked for many years. My brother had heart problems before. I have never felt anything like this before. It is hard for me to reach a hospital quickly from where I live in this rural area.
Doctor: I understand your symptoms are severe. Based on your description, I am concerned that you may be having a heart attack. I would like to ask you some more questions to confirm my suspicion. However, given the severity of your symptoms, I would recommend that you go to the nearest hospital ER immediately.
Patient: I’m not sure how to answer that, but my main